# Week 9 · Day 4 — The cut-off and watching the model

HW 9.4, Part B: five functions (TODO + an assert cell under each), the PSI on ten bins and the six-line note. Part A, `cutoff_table.sql`, goes into the 📎 box on the page; you can try it in section 7 here. Finish with **Kernel → Restart & Run All**.

In [1]:
from pathlib import Path
import hashlib
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")           # this notebook lives in data/python/week9/, so "../.." is the data/ folder
con = sqlite3.connect(DATA / "raw" / "nur.db")


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w9:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

In [2]:
# The scored loans and the scored applicants (as on the Day 4 page; run, do not edit)
S = pd.read_sql("""
SELECT l.loan_id, a.credit_score AS score,
       CASE WHEN l.status IN ('default', 'written_off') THEN 1 ELSE 0 END AS bad
FROM loans l JOIN applications a ON a.application_id = l.application_id
WHERE a.credit_score IS NOT NULL ORDER BY l.loan_id""", con)
s, y = S["score"].to_numpy(), S["bad"].to_numpy()
A = pd.read_sql("SELECT applied_at, credit_score FROM applications WHERE credit_score IS NOT NULL", con)
A["yr"], A["ym"] = A["applied_at"].str[:4], A["applied_at"].str[:7]
s24 = A.loc[A.yr == "2024", "credit_score"].to_numpy()
s26 = A.loc[A.yr == "2026", "credit_score"].to_numpy()
# W8D2's average Consumer loan (soum) and our LGD
ECON = {"interest": 5_695_580, "cof": 3_417_348, "opex": 823_325, "ead": 21_630_783, "lgd": 0.60}
BANDS = [560, 600, 640, 680, 720]
assert (len(S), y.sum(), len(s24), len(s26)) == (7615, 223, 2995, 5361)
print(len(S), "scored loans;", len(s24), "and", len(s26), "scored applicants in 2024 and 2026")

7615 scored loans; 2995 and 5361 scored applicants in 2024 and 2026


## 1. `cutoff_row(score, bad, cut, econ)` — one line of the strategy table
Approve `score >= cut`. A good loan earns interest − cof − opex; a bad one costs lgd × ead + cof + opex. Return a dict with `approved` (loans), `approval` (share), `bad_rate`, `avoided` (bads declined), `lost` (goods declined) and `profit` (soum).

In [3]:
def cutoff_row(score, bad, cut, econ):
    """approved, approval, bad_rate, avoided, lost and profit for the line 'approve score >= cut'."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [4]:
r = cutoff_row(s, y, 600, ECON)
check(r["approval"], ['0039d55e6a8d', '16cf1a023d03', '3c19b4eb8d71'], 4); check(r["bad_rate"], ['2ed6bb3354a2', '8bd8a691db35', 'cc96f3696c32'], 4); check(r["avoided"], ['106808f85ec8', 'e2b266ed340b', 'ea7ea20c161b'], 0); check(r["lost"], ['227133b48950', '75566bdd6e8f', 'c368484f66f3'], 0); check(r["profit"] / 1e9, ['6f220ab322bb', '897325e5450c', 'd9340a6a4b2f'], 2)
print("cutoff_row OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 2. `best_cutoff(score, bad, econ)` — the profit-maximising line
Try every cut-off from the lowest score to one above the highest; return `(cut, profit)`.

In [5]:
def best_cutoff(score, bad, econ):
    """(cut, profit) of the most profitable line 'approve score >= cut'."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [6]:
c, p = best_cutoff(s, y, ECON)
check(c, ['0230d4dff098', '69a5394c9a81', 'ed41f4a6632f'], 0); check(p / 1e9, ['4387604a512e', 'd9340a6a4b2f', 'e372e85b5998'], 2)
# the same with funding at 22 % a year: the cost of funds grows in proportion
ECON22 = dict(ECON, cof=ECON["cof"] * 22 / 18)
check(best_cutoff(s, y, ECON22)[0], ['3404530db3d7', '3a7f2d4c0892', '9ffbb4de9db8'], 0)
print("best_cutoff OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 3. `shares(x, edges)` — each bin's share
`np.digitize(x, edges)` numbers the bins 0 … len(edges); a value equal to an edge goes into the bin **above**.

In [7]:
def shares(x, edges):
    """The share of x in each of the len(edges) + 1 bins (a value on an edge goes up)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [8]:
assert np.allclose(shares(np.array([559, 560, 600, 720]), BANDS), [0.25, 0.25, 0.25, 0, 0, 0.25])
assert abs(shares(s24, BANDS).sum() - 1) < 1e-12
print("shares OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 4. `psi(expected, actual, floor=0.0001)` — the population stability index
Σ (actual − expected) × ln(actual ÷ expected), every share first raised to at least `floor` (an empty bin would make ln(0)).

In [9]:
def psi(expected, actual, floor=0.0001):
    """PSI of the actual shares against the expected ones, empty bins floored."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [10]:
check(psi([0.2, 0.8], [0.3, 0.7]), ['015a3b34b915', '5e612dee1bea', 'e02c4b3d84e3'], 4)
assert np.isfinite(psi([0.5, 0.5], [1.0, 0.0])), "an empty bin must not give infinity"
check(psi(shares(s24, BANDS), shares(s26, BANDS)), ['3df3d9f863e0', 'ca870b8fb380', 'e3a536f2da7d'], 4)
print("psi OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 5. `monthly_psi(frame, edges, ref)` — the monitoring line
One value per month (`ym`) of `frame`: the PSI of that month's scores against the reference shares `ref`.

In [11]:
def monthly_psi(frame, edges, ref):
    """A Series: the PSI of each month's credit scores against the reference shares."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [12]:
m = monthly_psi(A[A.yr == "2026"], BANDS, shares(s24, BANDS))
assert len(m) == 10
check(m.max(), ['67426c77f93e', '8c096f98c579', 'f456c4d28379'], 4); check(m.loc["2026-07"], ['584184790582', '622f598ce4c7', 'c42f8e4fca17'], 4)
print("monthly_psi OK")
m.round(4)

NotImplementedError: TODO: write me, then run the assert cell below

## 6. The PSI on ten bins (H3 on the page)
The edges are the nine deciles of 2024's scores. Expected = 2024's ten shares (about 10 % each), actual = 2026's.

In [13]:
edges10 = np.quantile(s24, np.arange(1, 10) / 10)
print(edges10)
# TODO: p10 = psi(... 2024's ten shares ..., ... 2026's ten shares ...)
p10 = None
print(p10)

[586.  611.  630.  647.6 663.  678.  694.  711.  737. ]
None


## 7. Try your `cutoff_table.sql` here (Part A)
Paste the query between the triple quotes and run: it must return 13 rows, 580 to 700.

In [14]:
sql = """
-- paste cutoff_table.sql here
"""
if "select" in sql.lower():
    display(pd.read_sql(sql, con))

## 8. The six-line note for Nilufar
The line · its approval and bad rate · its profit against approving everyone · the price of a 2 % target · the PSI · one trigger to revisit.

*(write here)*